# Phase 4 — Agent A5 : Noise Detector (PRD v3.0 — SVM first + CoT léger + Tools)
**YouTube Quality Analyzer** | Phase 4 — Implémentation

Ce notebook démontre l'agent A5 (`agents/a5_noise.py`) en version **PRD v3.0** :
- **SVM first** : `svm_spam_detector` + `count_repeated_chars` sur tous les commentaires
- **CoT léger** déclenché uniquement si confiance SVM < 0.75 (cas ambigus)
- 5 catégories de bruit : spam, offtopic, reaction_vide, toxique, bot
- Formule `Score_Bruit = (1 − noise_ratio%) / 100 × 100` = `100 − noise_ratio%`
- Pipeline anti-hallucination : `safe_llm_call` → `NoiseValidator` → retry ×3

> **Double mock requis** : même règle qu'A3/A4 — `agents.a5_noise.get_llm` + `models.llm_loader.get_llm`.  
> **Fallback SVM dynamique** : en l'absence de LLM, le score est calculé depuis les ratios SVM réels (pas un score fixe).

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))

from unittest.mock import MagicMock, patch
from agents.a5_noise import a5_noise

## 1. Corpus de test (avec bruit artificiel)

In [ ]:
CLEANED_COMMENTS = [
    {"text": "Great tutorial on machine learning!", "cleaned_text": "great tutorial on machine learning!"},
    {"text": "EARN 500€/DAY — click here: bit.ly/spam", "cleaned_text": "earn 500€/day — click here: bit.ly/spam"},
    {"text": "lol 😂😂😂", "cleaned_text": "lol 😂😂😂"},
    {"text": "This is completely off-topic but did you see the game last night?", "cleaned_text": "this is completely off-topic but did you see the game last night?"},
    {"text": "Very informative, learned about gradient descent.", "cleaned_text": "very informative, learned about gradient descent."},
    {"text": "SUBSCRIBE TO MY CHANNEL!!!", "cleaned_text": "subscribe to my channel!!!"},
]

STATE = {"cleaned_comments": CLEANED_COMMENTS}
print(f"{len(CLEANED_COMMENTS)} commentaires (dont 3 bruités attendus)")

## 2. A5 avec LLM mocké — corpus bruité

In [ ]:
# v3.0 — schéma JSON CoT léger (reasoning + svm_used)
mock_response = MagicMock()
mock_response.content = json.dumps({
    "reasoning": (
        "Thought 1: SVM detected 10% spam — confirms 'bit.ly/spam' and 'SUBSCRIBE' patterns. "
        "Thought 2: Off-topic (game comment) ~10%, reactions (lol 😂) ~15%. "
        "Thought 3: Overall noise_ratio = 0.40."
    ),
    "spam_ratio":     0.20,
    "offtopic_ratio": 0.10,
    "reaction_ratio": 0.15,
    "toxic_ratio":    0.0,
    "bot_ratio":      0.0,
    "noise_ratio":    0.40,
    "rationale": "Significant spam and off-topic content detected; reactions are empty.",
    "svm_used":  True,
})

mock_llm = MagicMock()
mock_llm.invoke.return_value = mock_response

# Double mock : garde du nœud + safe_llm_call (import local)
with patch("agents.a5_noise.get_llm", return_value=mock_llm), \
     patch("models.llm_loader.get_llm", return_value=mock_llm):
    result = a5_noise(STATE)

n = result["noise"]
print(f"spam_ratio      : {n['spam_ratio']:.1f}%")
print(f"offtopic_ratio  : {n['offtopic_ratio']:.1f}%")
print(f"reaction_ratio  : {n['reaction_ratio']:.1f}%")
print(f"toxic_ratio     : {n['toxic_ratio']:.1f}%")
print(f"bot_ratio       : {n['bot_ratio']:.1f}%")
print(f"noise_ratio     : {n['noise_ratio']:.1f}%  (= 40%)")
print(f"Score_Bruit     : {n['noise_score']:.1f} / 100  (= 100 - 40 = 60.0)")
print(f"svm_used        : {n.get('svm_used')}")
print(f"hallucination_flags : {result.get('hallucination_flags', [])}")
assert n["noise_score"] == 60.0
print("\nOK — formule 100 − noise_ratio% confirmée.")

## 3. Fallback — LLM indisponible

In [ ]:
# v3.0 — fallback SVM dynamique (pas de LLM)
with patch("agents.a5_noise.get_llm", return_value=None):
    fallback = a5_noise(STATE)

n = fallback["noise"]
print(f"Fallback noise_score : {n['noise_score']}")
print(f"Fallback spam_ratio  : {n.get('spam_ratio')}")
print(f"Fallback bot_ratio   : {n.get('bot_ratio')}")
print(f"Fallback svm_used    : {n.get('svm_used')}")

# v3.0 : fallback SVM-based — score calculé depuis les ratios réels (pas un score fixe 70.0)
# Formule : noise_r = spam_ratio*100 + bot_ratio*100*0.5 + 10 (base)
#           noise_score = 100 - noise_r
assert n.get("svm_used") is True, "svm_used doit être True en fallback SVM"
assert 0.0 <= n["noise_score"] <= 100.0, f"Score hors [0-100] : {n['noise_score']}"

# Vérification formule Score_Bruit
print("\nFormule (100 − noise_ratio%) :")
for nr in [0, 10, 26.7, 40, 60, 80, 100]:
    score = round(100.0 - nr, 2)
    print(f"  noise_ratio={nr:.1f}% → Score_Bruit={score:.1f}")
print("\nFallback v3.0 conforme (SVM dynamique).")

## Résumé A5 — PRD v3.0

| Comportement | Résultat |
|---|---|
| **SVM first** : `svm_spam_detector` + `count_repeated_chars` | Filtre déterministe sur 100% des commentaires |
| **CoT léger** | Déclenché si confiance SVM < 0.75 (cas ambigus) |
| `reasoning` | Pensée CoT : Thought 1→2→3 → ratios finaux |
| `svm_used` | Toujours `True` (SVM toujours exécuté) |
| 5 catégories de bruit détectées | spam, offtopic, reaction_vide, toxique, bot |
| Formule Score_Bruit | `100 − noise_ratio%` |
| `noise_ratio=40%` → `Score=60.0` | Confirmé |
| `hallucination_flags` | Propagés dans le state LangGraph |
| LLM indisponible | Fallback **SVM dynamique** — score calculé depuis ratios réels |
| Double mock requis | `agents.a5_noise.get_llm` + `models.llm_loader.get_llm` |

> **Poids w3 = 0.25** — dimension la moins influente, mais garde-fou contre les corpus pollués  
> **Changement v3.0** : le fallback n'est plus `noise_score=70.0` fixe — il dépend des ratios SVM détectés